In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/salary_analysis.db")
df = pd.read_sql_query("SELECT * FROM salary_data;", conn)
conn.close()

df.head()


In [ ]:
import plotly.express as px

fig = px.histogram(df, x="salary_in_usd", nbins=50, 
                   title="Salary Distribution (USD)",
                   color_discrete_sequence=["#636EFA"])
fig.update_layout(bargap=0.1)
fig.show()


In [ ]:
fig = px.box(df, x="experience_level", y="salary_in_usd",
             color="experience_level",
             title="Salary by Experience Level")
fig.show()


In [ ]:
remote_avg = df.groupby("remote_ratio")["salary_in_usd"].mean().reset_index()
fig = px.bar(remote_avg, x="remote_ratio", y="salary_in_usd",
             color="remote_ratio", title="Average Salary by Remote Work Type")
fig.show()


In [ ]:
import plotly.express as px

fig = px.scatter(
    df,
    x="Cost of Living Index",
    y="salary_in_usd",
    color="experience_level",
    size="salary_in_usd",
    hover_data=["Country", "employment_type"],
    title="Salary vs Cost of Living by Experience"
)
fig.show()


In [ ]:
country_avg = (
    df.groupby("Country")["salary_in_usd"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

fig = px.bar(
    country_avg,
    x="salary_in_usd",
    y="Country",
    orientation="h",
    color="salary_in_usd",
    color_continuous_scale="Viridis",
    title="Top 10 Countries by Average Salary"
)
fig.show()


In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


In [ ]:
import numpy as np

corr = df.corr(numeric_only=True)
fig = px.imshow(corr, text_auto=True, aspect="auto",
                title="Feature Correlation Heatmap",
                color_continuous_scale="RdBu_r")
fig.show()


In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import sqlite3

# Load data
conn = sqlite3.connect("../data/salary_analysis.db")
df = pd.read_sql_query("SELECT * FROM salary_data;", conn)
conn.close()
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Check your column names (just to confirm)
print(df.columns.tolist())

# Build app
app = Dash(__name__)

app.layout = html.Div([
    html.H1("?? Global Salary Dashboard", style={"textAlign": "center"}),
    
    html.Label("Select a Country:", style={"fontWeight": "bold"}),
    dcc.Dropdown(
        options=[{"label": c, "value": c} for c in df["country"].unique()],
        value="United States",
        id="country-filter"
    ),
    
    dcc.Graph(id="salary-dist")
])

@app.callback(
    Output("salary-dist", "figure"),
    Input("country-filter", "value")
)
def update_chart(country):
    dff = df[df["country"] == country]
    fig = px.histogram(
        dff, 
        x="salary_in_usd",
        nbins=40,
        title=f"Salary Distribution in {country}",
        color_discrete_sequence=["#7d0be1ff"]
    )
    fig.update_layout(
        xaxis_title="Salary (USD)",
        yaxis_title="Frequency",
        template="plotly_white"
    )
    return fig

if __name__ == "__main__":
    app.run(debug=True)
